# VOC Sedici extract

Notebook de prueba para el nodo `extract_vocsedici_tables` del pipeline `extract_vocsedici`.

Lee tablas desde `voc_mariadb/{table}`, agrega metadata de extraccion y deja los DataFrames listos para persistir como `raw/voc/{table}#parquet` en el pipeline. Esta notebook no hace `catalog.save`.

In [ ]:
%load_ext kedro.ipython


In [ ]:
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 10)


In [ ]:
tables = catalog.load("params:extract_vocsedici_options.tables")
source_label = catalog.load("params:extract_vocsedici_options.source_label")
extract_env = catalog.load("params:extract_vocsedici_options.env")
filter_param = catalog.load("params:extract_vocsedici_options.filter_param")
filter_value = catalog.load("params:extract_vocsedici_options.filter_value")

tables


In [ ]:
dataframes = [catalog.load(f"voc_mariadb/{table}") for table in tables]

[(table, df.shape) for table, df in zip(tables, dataframes)]


## Funcion completa

Esta celda replica la implementacion del nodo `extract_vocsedici_tables`. La idea es editar aca primero y despues copiar a `src/kedro_cic/pipelines/extract_vocsedici/nodes.py`.

In [ ]:
VOCSEDICI_TABLES = (
    "node",
    "node_field_data",
    "paragraphs_item",
    "paragraphs_item_field_data",
    "paragraph__field_persona_id",
    "paragraph__field_institucion",
    "paragraph__field_fecha_inicio",
    "paragraph__field_fecha_fin",
    "node__field_nombre",
    "node__field_apellido",
    "node__field_orcid",
    "node__field_mail",
    "node__field_dni",
    "node__field_cuit",
    "node__field_telefono",
    "node__field_direcci_n",
    "node__field_google_scholar",
    "node__field_researchgate",
    "node__field_old_id",
    "node__field_filiacion",
    "node__field_nombre_institucion",
    "node__field_nombre_institucion_variant",
    "node__field_abreviatura",
    "node__field_id_pidu",
    "node__field_id_termino",
    "node__field_padre",
)


def _add_extract_metadata(
    df: pd.DataFrame,
    *,
    source_label: str,
    extract_env: str,
    source_table: str,
    filter_param: str | None,
    filter_value,
    extract_datetime: pd.Timestamp,
) -> pd.DataFrame:
    enriched_df = df.copy()
    enriched_df["_source_system"] = "voc"
    enriched_df["_source_table"] = source_table
    enriched_df["_extract_datetime"] = extract_datetime
    enriched_df["_extract_date"] = extract_datetime.date()
    enriched_df["_source_label"] = source_label
    enriched_df["_extract_env"] = extract_env
    enriched_df["_filter_param"] = filter_param if filter_param else pd.NA
    enriched_df["_filter_value"] = filter_value if filter_value not in (None, "") else pd.NA
    return enriched_df


def extract_vocsedici_tables(
    tables,
    source_label,
    extract_env,
    filter_param,
    filter_value,
    *dataframes,
):
    configured_tables = tuple(tables)

    if configured_tables != VOCSEDICI_TABLES:
        raise ValueError(
            f"Configured tables do not match pipeline inputs. "
            f"Expected {VOCSEDICI_TABLES}, got {configured_tables}."
        )

    if len(dataframes) != len(VOCSEDICI_TABLES):
        raise ValueError(
            f"Expected {len(VOCSEDICI_TABLES)} dataframes, got {len(dataframes)}."
        )

    extract_datetime = pd.Timestamp.now(tz="UTC").floor("s").tz_localize(None)

    return tuple(
        _add_extract_metadata(
            df,
            source_label=source_label,
            extract_env=extract_env,
            source_table=table_name,
            filter_param=filter_param,
            filter_value=filter_value,
            extract_datetime=extract_datetime,
        )
        for table_name, df in zip(VOCSEDICI_TABLES, dataframes)
    )


In [ ]:
extracted_tables = extract_vocsedici_tables(
    tables,
    source_label,
    extract_env,
    filter_param,
    filter_value,
    *dataframes,
)

summary = pd.DataFrame(
    {
        "table": tables,
        "rows": [len(df) for df in extracted_tables],
        "columns": [len(df.columns) for df in extracted_tables],
        "source_system": [df["_source_system"].iloc[0] if len(df) else "voc" for df in extracted_tables],
        "source_label": [df["_source_label"].iloc[0] if len(df) else source_label for df in extracted_tables],
        "extract_datetime": [df["_extract_datetime"].iloc[0] if len(df) else pd.NA for df in extracted_tables],
    }
)

summary


## Inspeccion rapida

Usar `table_to_preview` para revisar una tabla puntual antes de copiar cambios a `nodes.py`.

In [ ]:
table_to_preview = "node"
df_preview = extracted_tables[tables.index(table_to_preview)]

df_preview.head()


In [ ]:
metadata_columns = [
    "_source_system",
    "_source_table",
    "_extract_datetime",
    "_extract_date",
    "_source_label",
    "_extract_env",
    "_filter_param",
    "_filter_value",
]

df_preview[metadata_columns].head()
